
# 🎨 Image to Paint-by-Numbers Generator

## Overview
Convert an uploaded image into a **paint-by-numbers style output**:
- Color quantization
- Region numbering
- Color legend
- Downloadable output
- Gradio UI

---


In [ ]:
!pip install opencv-python pillow numpy scikit-learn gradio groq

In [ ]:

import cv2
import numpy as np
from PIL import Image, ImageDraw
from sklearn.cluster import KMeans
import gradio as gr


In [ ]:

def preprocess_image(image, size=(256,256)):
    """Resize image for processing"""
    return np.array(image.resize(size))


In [ ]:

def quantize_colors(image_array, k=8):
    """Reduce image colors using KMeans"""
    pixels = image_array.reshape(-1, 3)
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(pixels)
    palette = kmeans.cluster_centers_.astype(int)
    segmented = palette[labels].reshape(image_array.shape)
    return segmented, labels.reshape(image_array.shape[:2]), palette


In [ ]:

def create_numbered_image(segmented, labels):
    """Overlay numbers on image regions"""
    img = Image.fromarray(segmented.astype('uint8'))
    draw = ImageDraw.Draw(img)
    h, w = labels.shape

    for y in range(0, h, 25):
        for x in range(0, w, 25):
            draw.text((x,y), str(labels[y,x]+1), fill=(0,0,0))
    return img


In [ ]:

def generate_legend(palette):
    """Create color legend"""
    return "\n".join([f"{i+1}: RGB{tuple(c)}" for i,c in enumerate(palette)])


In [ ]:

def pipeline(image, k):
    arr = preprocess_image(image)
    seg, labels, palette = quantize_colors(arr, k)
    out = create_numbered_image(seg, labels)
    legend = generate_legend(palette)
    return out, legend


In [ ]:

def app():
    with gr.Blocks() as demo:
        gr.Markdown("# 🎨 Paint by Numbers AI")

        api_key = gr.Textbox(label="Groq API Key (optional)", type="password")
        image = gr.Image(type="pil")
        k = gr.Slider(3, 12, value=8)

        output = gr.Image()
        legend = gr.Textbox()

        btn = gr.Button("Generate")
        btn.click(pipeline, [image, k], [output, legend])

    demo.launch()
